# Explore Data and Filter Out Non-PII (Plain Python)

## Import packages

In [2]:
from pathlib import Path
import re

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

## Helper functions

In [3]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def save_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df.to_csv(path, index=False)
        return
    if suffix in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
        return
    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True)
        return
    if suffix == ".json":
        df.to_json(path, orient="records")
        return
    if suffix in {".pkl", ".pickle"}:
        df.to_pickle(path)
        return

    raise ValueError(f"Unsupported output file type: {suffix}")


def as_text_series(series):
    return series.fillna("").astype(str)


def top_value_counts(df, column, n):
    return (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
        .head(n)
    )


def regex_match_counts(df, column, patterns):
    values = as_text_series(df[column])
    counts = []

    for pattern in patterns:
        regex = re.compile(pattern)
        match_count = values.map(lambda value: bool(regex.search(value))).sum()
        counts.append({"pattern": pattern, "count": int(match_count)})

    return pd.DataFrame(counts)


def filter_out_regex_patterns(df, column, patterns):
    values = as_text_series(df[column])
    matches_any_pattern = pd.Series(False, index=df.index)

    for pattern in patterns:
        regex = re.compile(pattern)
        matches_any_pattern |= values.map(
            lambda value: bool(regex.search(value))
        )

    return df.loc[~matches_any_pattern].copy()

## Set parameters
Load raw data as a pandas DataFrame and set the name of the target column to be analyzed for PII.

In [4]:
# Replace these paths and column names for your local data.
data_path = Path("data/pii_finetune_data_formatted.json")
target_col = "text"
output_path = Path("data/filtered_output.csv")

data = load_dataframe(data_path)

In [5]:
# # Replace these paths and column names for your local data.
# data_path = Path("path/to/input.csv")
# target_col = "MY_TARGET"
# output_path = Path("path/to/output.csv")

# data = load_dataframe(data_path)


## Basic Exploration

In [ ]:
# Show columns and data types.
display(data.dtypes.rename("dtype").to_frame())
display(data.head())

In [ ]:
# Show row count.
print(f"Row count: {len(data):,}")

In [ ]:
# Show empty string target column row count.
empty_string_count = int(data[target_col].eq("").sum())
print(f"Empty string {target_col} rows: {empty_string_count:,}")

In [ ]:
# Show null target column row count.
null_count = int(data[target_col].isna().sum())
print(f"Null {target_col} rows: {null_count:,}")

In [ ]:
# See how many distinct non-null values exist for the target column.
distinct_count = int(data[target_col].nunique(dropna=True))
print(f"Distinct non-null {target_col} values: {distinct_count:,}")

In [ ]:
# See 100 most common values for the target column.
# Inspect visually for common non-PII patterns.
display(top_value_counts(data, target_col, 100))

## Round 1: Remove Patterns that are Unlikely to Have PII

In [12]:
# List of non-PII regex patterns.
# Adjust for your use case if not analyzing lab test results.
non_pii_patterns = [
    r"^-?[0-9]+([.][0-9]+)?$",  # Number
    r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$",  # Date (MM/DD/YYYY)
    r"^[0-9]{2}/[0-9]{2}/[0-9]{2}$",  # Date (MM/DD/YY)
    r"^-?[0-9]+([.][0-9]+)?[ ]*?%$",  # Number + %
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?x10e3/uL$",  # Number + x10e3/uL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mg/dL$",  # Number + mg/dL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?g/dL$",  # Number + g/dL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?ng/mL$",  # Number + ng/mL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mmol/L$",  # Number + mmol/L
    r"^[A-TV-Z][0-9][0-9AB]\.?[0-9A-TV-Z]{0,4}$",  # ICD-10 code
]

In [ ]:
# Show how many rows match each pattern.
display(regex_match_counts(data, target_col, non_pii_patterns))

In [ ]:
# Filter out 1- and 2-character records, then remove regex matches.
long_enough_target = as_text_series(data[target_col]).str.len() > 2
data_filtered_1 = data.loc[long_enough_target].copy()
data_filtered_1 = filter_out_regex_patterns(
    data_filtered_1, target_col, non_pii_patterns
)

display(data_filtered_1[[target_col]].head())
print(f"Rows left after 1st filter: {len(data_filtered_1):,}")

## Round 2: Remove Additional Patterns that are Unlikely to Have PII

In [ ]:
# Inspect non-PII patterns in data left after filter 1.
# See 100 most common values in target column.
display(top_value_counts(data_filtered_1, target_col, 100))

In [16]:
# List of additional non-PII regex patterns.
# Adjust for your use case if not analyzing lab test results.
non_pii_patterns = [
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?IU/L$",  # Number + IU/L
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?U/L$",  # Number + U/L
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?fL$",  # Number + fL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mL/min/1\.73$",  # Number + mL/min/1.73
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?pg$",  # Number + pg
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?(calc)$",  # Number + calc
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?\(calc\)$",  # Number + (calc)
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?K/uL$",  # Number + K/uL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?IU/mL$",  # Number + IU/mL
    r"^>[ ]*-?[0-9]+([.][0-9]+)?$",  # > number
    r"^<[ ]*-?[0-9]+([.][0-9]+)?$",  # < number
]

In [ ]:
# Show how many rows match each pattern.
display(regex_match_counts(data_filtered_1, target_col, non_pii_patterns))

In [ ]:
# Apply 2nd set of filters.
data_filtered_2 = filter_out_regex_patterns(
    data_filtered_1, target_col, non_pii_patterns
)

display(data_filtered_2[[target_col]].head())
print(f"Rows left after 2nd filter: {len(data_filtered_2):,}")

## Round 3: Remove Common Terms that are Non-PII

In [ ]:
# Inspect data left after filter 2.
# See 1000 most common values for the target column. Edit number as needed.
top_1000_terms = top_value_counts(data_filtered_2, target_col, 1)
display(top_1000_terms)

In [ ]:
# If all 1000 terms look PII free, filter them out of the dataset as well.
# If there is some PII in the list, edit top_1000_terms before running this cell.
top_1000_values = top_1000_terms[target_col]
data_filtered_3 = data_filtered_2.loc[
    ~data_filtered_2[target_col].isin(top_1000_values)
].copy()

print(f"Rows left after 3rd filter: {len(data_filtered_3):,}")

Conduct more rounds of exploration and filtering as desired.

## Save Final Filtered Data

In [ ]:
save_dataframe(data_filtered_3, output_path)
print(f"Saved {len(data_filtered_3):,} rows to {output_path.resolve()}")